In [1]:
using LowLevelFEM

[ Info: Waiting for another process (pid: 2163345) to finish precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07]. Pidfile: /home/perebal/.julia/compiled/v1.12/LowLevelFEM/Gjcu1_pM7Iw.ji.pidfile

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
nel = 50
A = 1
E = 1.0
ρ = 1.0

n = 1000

model = line_mesh(n=nel)

material = Material("body", E=E, ρ=ρ)

Pu = Problem([material], type=:ScalarField, dim=1, field=:u, rhs_field=:f)

K = ∫(AxialGrad(Pu) ⋅ AxialGrad(Pu) * (E * A))
M = ∫(Pu ⋅ Pu * (ρ * A))
M = consistentToLumped(M)

bc = BoundaryCondition("left", u=0)

Tmin = smallestPeriodTime(K, M, support=[bc])
dt = Tmin / π
tmax = n * dt

g = ScalarField(Pu, "left", (x, y, z, t) -> 0.1 * sin(10t), steps=n, tmax=tmax)
h = ScalarField(Pu, "right", (x, y, z, t) -> 0.5 * sin(5t), steps=n, tmax=tmax)

f = ScalarField[]
for i in 1:n
    hi = ScalarField(h, step=i)
    fi = ∫(Pu ⋅ hi, Γ="right")
    push!(f, fi)
end
f = mergeFields(f)

bc = BoundaryCondition("left", u=g)

u0 = scalarField(Pu, "body", 0)
v0 = scalarField(Pu, "body", 0)

u1, v1 = CDM(K, M, f, u0, v0, n, dt, support=[bc])
u2, v2 = HHT(K, M, f, u0, v0, n, dt, support=[bc])

(ScalarField(Matrix{Float64}[], [0.0 0.01988897441626565 … -0.09449619162219774 -0.08610105314097571; 0.0 0.00035342541286900817 … 0.08080711000935173 0.09687323132695172; … ; 0.0 1.040750380476015e-5 … 0.09219143419010375 0.10191652237513638; 0.0 6.06487949520279e-5 … 0.08657636054846192 0.10072358873786866], [0.0, 0.020002467654795218, 0.040004935309590435, 0.060007402964385656, 0.08000987061918087, 0.10001233827397608, 0.12001480592877131, 0.14001727358356653, 0.16001974123836174, 0.18002220889315695  …  19.802442978247264, 19.822445445902062, 19.842447913556857, 19.86245038121165, 19.882452848866446, 19.90245531652124, 19.922457784176036, 19.94246025183083, 19.96246271948563, 19.982465187140424], Int64[], 1000, :scalar, Problem("line_mesh", :ScalarField, 1, 1, Material[Material("body", :Hooke, 1.0, 0.3, 0.5769230769230768, 0.3846153846153846, 0.8333333333333333, 1.0, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 51, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, noth

In [3]:
showDoFResults(g, name="g")
u01 = showDoFResults(u1)
u02 = showDoFResults(u2)
plotOnPath("body", u01)
plotOnPath("body", u02)

4

In [4]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
